In [ ]:
# Trying my own history!!

import datetime

import requests
from bs4 import BeautifulSoup
import pandas as pd

session = requests.Session()

period=364 # max for 364 days!
end = datetime.datetime.now()
start = end - datetime.timedelta(days=period)


In [ ]:
# For NIFTY index

from pathlib import Path
import yaml

url='http://www1.nseindia.com/products/dynaContent/equities/indices/historicalindices.jsp'

# Header from yml files
root = Path().cwd() 
with open(root / "conf" / "config.yml", 'r') as f:
    config = yaml.safe_load(f)

headers = config['headers']

params={'indexType': 'NIFTY 50', 'fromDate': start.strftime('%d-%m-%Y'), 'toDate': end.strftime('%d-%m-%Y')}


# headers =  {'Connection': 'keep-alive',
#             'Cache-Control': 'max-age=0',
#             'DNT': '1',
#             'Upgrade-Insecure-Requests': '1',
#             'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/79.0.3945.79 Safari/537.36',
#             'Sec-Fetch-User': '?1',
#             'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
#             'Sec-Fetch-Site': 'none',
#             'Sec-Fetch-Mode': 'navigate',
#             'Accept-Encoding': 'gzip, deflate, br',
#             'Accept-Language': 'en-US,en;q=0.9,hi;q=0.8'}

resp = session.get(url=url, params=params, headers=headers)
soup = BeautifulSoup(resp.text, 'lxml')
table = soup.find_all('table')

try:
    df = pd.read_html(str(table))[0]
except ValueError as e:
    print(f"illformed table!! {e}")

# cleaning up the table
cols = [list(c)[-1] for c in df.columns]
df.columns = cols

df[df.columns[0]] = pd.to_datetime(df.Date, errors='coerce') # coerces last row to NaT
df = df.dropna()
df[df.columns[1:]] = df[df.columns[1:]].apply(pd.to_numeric) # converts non-date columns to numeric


In [ ]:
d = {'indices': {"NIFTY": "NIFTY 50",
               "BANKNIFTY": "NIFTY BANK",
               "NIFTYINFRA": "NIFTY INFRA",
               "NIFTYIT": "NIFTY IT",
               "NIFTYMID50": "NIFTY MIDCAP 50",
               "NIFTYPSE": "NIFTY PSE"}}

import ruamel.yaml
import sys

yaml = ruamel.yaml.YAML()
yaml.default_flow_style = None
yaml.dump(d, sys.stdout)

In [ ]:
yaml?

In [ ]:
import sys
import ruamel.yaml


windows_list = []
server_list = ['abc-def-01', 'pqr-str-02']
site_list = ['dev', 'prod']

server_dict = dict(zip(server_list, site_list))

for k,v in server_dict.items():
    a = { 'SERVER': k, 'SITE': v }
    windows_list.append(a)

final_dict = {'service':'something', 'servers': windows_list}

yaml = ruamel.yaml.YAML()
yaml.default_flow_style = None
yaml.dump(final_dict, sys.stdout)